#  Closed Deals - Bronze Ingestion


## Imports

In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, TimestampType, BooleanType, StructField, DecimalType, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_closed_deals"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist_marketing"
source_dataset = "closed_deals"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("mql_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("sdr_id", StringType(), True),
    StructField("sr_id", StringType(), True),
    StructField("won_date", TimestampType(), True),
    StructField("business_segment", StringType(), True),
    StructField("lead_type", StringType(), True),
    StructField("lead_behaviour_profile", StringType(), True),
    StructField("has_company", BooleanType(), True),
    StructField("has_gtin", BooleanType(), True),
    StructField("average_stock", StringType(), True),
    StructField("business_type", StringType(), True),
    StructField("declared_product_catalog_size", DecimalType(10, 1), True),
    StructField("declared_monthly_revenue", DecimalType(18, 2), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

In [0]:
display(spark.table(target_table).limit(5))

In [0]:
spark.table(target_table).count()